# Simple Exponential Smoothing (SES)

Simple Exponential Smoothing is the foundation of the entire exponential
smoothing family.  The core idea is beautifully simple: **forecast the future
as a weighted average of all past observations, where recent observations
receive exponentially more weight than distant ones.**

SES is appropriate for data with **no trend** and **no seasonality** — just
random fluctuations around a slowly changing level.  Despite its simplicity,
SES often performs surprisingly well and serves as the building block for
Holt's method (adds trend) and Holt-Winters (adds seasonality).

**Key references:**
- FPP3 Section 8.1–8.2
- Portilla TSA Section 05: Introduction to Exponential Smoothing

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

## 1. Load Data — Daily Total Female Births

We use the daily female births dataset (365 days, 1959) because it has
**no trend and no seasonality** — exactly the setting where SES is designed
to work.  The series fluctuates around a roughly constant level.

In [ ]:
births = pd.read_csv(
    DATA_DIR / "DailyTotalFemaleBirths.csv",
    index_col="Date",
    parse_dates=True,
)
births.columns = ["Births"]

print(f"Shape: {births.shape}")
print(f"Date range: {births.index[0].date()} to {births.index[-1].date()}")
births.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(births["Births"], color="tab:blue", alpha=0.8)
ax.axhline(births["Births"].mean(), color="tab:red", linestyle="--", alpha=0.6,
           label=f"Mean = {births['Births'].mean():.1f}")
ax.set_title("Daily Total Female Births (1959)")
ax.set_ylabel("Number of Births")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("No obvious trend or seasonality — the data fluctuates around a")
print("roughly constant level.  This is ideal for SES.")

## 2. Train / Test Split

In [ ]:
TRAIN_SIZE = 335
train = births.iloc[:TRAIN_SIZE]
test = births.iloc[TRAIN_SIZE:]
HORIZON = len(test)

print(f"Training: {train.index[0].date()} to {train.index[-1].date()} ({len(train)} days)")
print(f"Test:     {test.index[0].date()} to {test.index[-1].date()} ({len(test)} days)")

In [ ]:
fig, ax = plt.subplots()
ax.plot(train["Births"], label="Train")
ax.plot(test["Births"], label="Test", color="tab:orange")
ax.axvline(test.index[0], color="grey", linestyle="--", alpha=0.7)
ax.set_title("Daily Births — Train / Test Split")
ax.set_ylabel("Number of Births")
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. The Weighted Average Form

The SES forecast can be written as a **weighted average of all past
observations**:

$$
\hat{y}_{T+1|T} = \alpha y_T + \alpha(1-\alpha) y_{T-1}
+ \alpha(1-\alpha)^2 y_{T-2} + \cdots
$$

where $0 \le \alpha \le 1$ is the **smoothing parameter**.  The weights
$\alpha, \alpha(1-\alpha), \alpha(1-\alpha)^2, \ldots$ are geometric
and sum to 1.  Recent observations get the most weight, and the weights
decay **exponentially** — hence the name.

In [ ]:
# Visualize the exponential decay of weights for different alpha values
fig, ax = plt.subplots(figsize=(10, 5))
lags = np.arange(20)

for alpha in [0.1, 0.3, 0.5, 0.8, 0.95]:
    weights = alpha * (1 - alpha) ** lags
    ax.plot(lags, weights, marker="o", markersize=4, label=f"alpha = {alpha}")

ax.set_title("Exponential Decay of Weights by Alpha")
ax.set_xlabel("Lag (periods into the past)")
ax.set_ylabel("Weight")
ax.legend()
plt.tight_layout()
plt.show()

print("High alpha: most weight on recent data (fast adaptation).")
print("Low alpha:  weight spread across many past observations (slow adaptation).")

In [ ]:
# Verify that the weights sum to (approximately) 1
for alpha in [0.1, 0.3, 0.5, 0.8, 0.95]:
    weights_100 = alpha * (1 - alpha) ** np.arange(100)
    print(f"alpha = {alpha:.2f}  |  sum of first 100 weights = {weights_100.sum():.6f}")

print("\nThe weights are geometric and sum to 1 in the infinite limit.")

---
## 4. The Component Form

The equivalent **component form** is more practical for computation:

$$
\text{Forecast equation:} \quad \hat{y}_{t+h|t} = \ell_t
$$

$$
\text{Level equation:} \quad \ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}
$$

- $\ell_t$ is the **level** (smoothed value) at time $t$.
- The forecast for any future horizon $h$ is simply the last level $\ell_t$.
- The level is updated each period as a weighted combination of the new
  observation $y_t$ and the previous level $\ell_{t-1}$.

This is a **flat forecast function**: all future forecasts are equal to the
last estimated level, regardless of the horizon $h$.

In [ ]:
def ses_from_scratch(y, alpha, initial_level=None):
    """Simple Exponential Smoothing implemented from scratch.

    Parameters
    ----------
    y : array-like
        The time series values.
    alpha : float
        Smoothing parameter (0 < alpha < 1).
    initial_level : float, optional
        Starting level.  Defaults to the first observation.

    Returns
    -------
    levels : np.ndarray
        The smoothed level at each time step.
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    levels = np.zeros(n)

    # Initialize
    levels[0] = initial_level if initial_level is not None else y[0]

    # Recursion
    for t in range(1, n):
        levels[t] = alpha * y[t] + (1 - alpha) * levels[t - 1]

    return levels


print("ses_from_scratch() defined.")

In [ ]:
# Apply SES from scratch with different alpha values
y_train = train["Births"].values

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, y_train, color="black", alpha=0.4, label="Actual")

for alpha in [0.05, 0.2, 0.5, 0.9]:
    levels = ses_from_scratch(y_train, alpha)
    ax.plot(train.index, levels, label=f"alpha = {alpha}", linewidth=1.5)

ax.set_title("SES Smoothed Levels — Effect of Alpha")
ax.set_ylabel("Number of Births")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("alpha = 0.05: very smooth — the level barely reacts to new data.")
print("alpha = 0.90: very responsive — the level tracks the data closely.")

---
## 5. Alpha Parameter Interpretation

| Alpha | Behavior | Memory | Use when... |
|-------|----------|--------|-------------|
| Close to 0 | Slow learning | Long (many past obs matter) | Level changes slowly |
| Close to 1 | Fast learning | Short (only recent obs matter) | Level changes rapidly |
| = 0 | Forecast = initial level (ignores data) | Infinite | Never (degenerate) |
| = 1 | Forecast = last observation (naive method) | None | Random walk data |

The optimal alpha is typically estimated by minimizing the sum of squared
one-step-ahead forecast errors (SSE).

In [ ]:
# Show the relationship between alpha and forecast behavior
# Forecast = last level, applied to the test period
fig, ax = plt.subplots(figsize=(14, 5))

# Plot test actuals
ax.plot(train["Births"].iloc[-60:], color="black", alpha=0.4, label="Train (last 60 days)")
ax.plot(test["Births"], color="tab:blue", linewidth=2, label="Actual (test)")

for alpha, color in [(0.05, "tab:red"), (0.3, "tab:green"), (0.7, "tab:purple"), (0.95, "tab:orange")]:
    levels = ses_from_scratch(y_train, alpha)
    last_level = levels[-1]
    # SES forecast is flat at the last level
    ax.axhline(last_level, color=color, linestyle="--", alpha=0.7,
               label=f"alpha={alpha} -> forecast={last_level:.1f}")

ax.set_title("SES Flat Forecasts for Different Alpha Values")
ax.set_ylabel("Number of Births")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("All SES forecasts are flat lines — they differ only in their level.")
print("The level depends on how much weight alpha gives to recent vs distant observations.")

---
## 6. Optimization — Finding the Best Alpha

The optimal values of $\alpha$ and the initial level $\ell_0$ are found by
minimizing the **sum of squared errors** (SSE) of the one-step-ahead
forecasts on the training data:

$$
\text{SSE} = \sum_{t=1}^{T} (y_t - \hat{y}_{t|t-1})^2
= \sum_{t=1}^{T} (y_t - \ell_{t-1})^2
$$

In [ ]:
def ses_sse(y, alpha, initial_level=None):
    """Compute the SSE for SES with given alpha and initial level."""
    y = np.asarray(y, dtype=float)
    n = len(y)
    level = initial_level if initial_level is not None else y[0]
    sse = 0.0

    for t in range(1, n):
        error = y[t] - level
        sse += error ** 2
        level = alpha * y[t] + (1 - alpha) * level

    return sse


# Grid search over alpha
alphas = np.linspace(0.01, 0.99, 99)
sse_values = [ses_sse(y_train, a) for a in alphas]

best_alpha = alphas[np.argmin(sse_values)]
print(f"Optimal alpha (grid search): {best_alpha:.2f}")
print(f"Minimum SSE: {min(sse_values):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(alphas, sse_values, color="tab:blue")
ax.axvline(best_alpha, color="tab:red", linestyle="--",
           label=f"Best alpha = {best_alpha:.2f}")
ax.set_title("SSE vs Alpha — Finding the Optimal Smoothing Parameter")
ax.set_xlabel("Alpha")
ax.set_ylabel("Sum of Squared Errors (SSE)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. statsmodels Implementation

In practice, we use `statsmodels.tsa.holtwinters.SimpleExpSmoothing` which
handles parameter optimization automatically via maximum likelihood
estimation.

In [ ]:
model = SimpleExpSmoothing(
    train["Births"],
    initialization_method="estimated",
)
fit = model.fit(optimized=True)

print(f"Optimal alpha (statsmodels): {fit.params['smoothing_level']:.4f}")
print(f"Initial level:              {fit.params['initial_level']:.4f}")
print(f"SSE:                        {fit.sse:.2f}")
print(f"AIC:                        {fit.aic:.2f}")

In [ ]:
fit.summary()

In [ ]:
# Generate forecasts
forecast = fit.forecast(HORIZON)

print(f"SES forecast (flat): {forecast.iloc[0]:.2f}")
print(f"All forecasts equal? {forecast.nunique() == 1}")
print(f"\nFirst 5 forecast values:")
print(forecast.head())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Births"], label="Train", color="black", alpha=0.5)
ax.plot(test["Births"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(forecast, label="SES forecast", color="tab:red", linestyle="--", linewidth=2)
ax.axvline(test.index[0], color="grey", linestyle=":", alpha=0.5)
ax.set_title("Simple Exponential Smoothing — Forecast")
ax.set_ylabel("Number of Births")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

print("The SES forecast is a flat line at the last estimated level.")
print("This is the defining characteristic of SES: no trend, no seasonality.")

---
## 8. Fitted Values

The **fitted values** are the one-step-ahead in-sample forecasts:
$\hat{y}_{t|t-1} = \ell_{t-1}$.  They show how well SES tracks the
training data.

In [ ]:
fitted_vals = fit.fittedvalues

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Births"], label="Actual", color="tab:blue", alpha=0.7)
ax.plot(fitted_vals, label="Fitted (SES)", color="tab:red", alpha=0.7, linestyle="--")
ax.set_title("SES Fitted Values (One-Step-Ahead In-Sample Forecasts)")
ax.set_ylabel("Number of Births")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals
residuals = train["Births"] - fitted_vals

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(residuals, color="tab:blue", alpha=0.7)
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set_title("SES Residuals Over Time")
axes[0].set_ylabel("Residual")

axes[1].hist(residuals.dropna(), bins=25, edgecolor="black", alpha=0.7)
axes[1].set_title("Distribution of SES Residuals")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.4f} (should be near 0)")
print(f"Std residual:  {residuals.std():.4f}")

---
## 9. Comparing Alpha Values with statsmodels

Let us fit SES with several fixed alpha values to see how the smoothed
level and forecast change.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Births"], color="black", alpha=0.3, label="Actual")

results = []
for alpha in [0.1, 0.3, 0.5, 0.8]:
    m = SimpleExpSmoothing(train["Births"], initialization_method="estimated")
    f = m.fit(smoothing_level=alpha, optimized=False)
    fc = f.forecast(HORIZON)
    ax.plot(f.fittedvalues, label=f"alpha={alpha} (level)", linewidth=1.2)
    ax.plot(fc, linestyle="--", linewidth=1.5)
    results.append({"alpha": alpha, "SSE": f.sse, "forecast": fc.iloc[0]})

ax.plot(test["Births"], color="tab:blue", linewidth=2, label="Actual (test)")
ax.axvline(test.index[0], color="grey", linestyle=":", alpha=0.5)
ax.set_title("SES with Different Fixed Alpha Values")
ax.set_ylabel("Number of Births")
ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
results_df = pd.DataFrame(results)
results_df["SSE"] = results_df["SSE"].round(2)
results_df["forecast"] = results_df["forecast"].round(2)
print(results_df.to_string(index=False))

---
## 10. SES vs Naive and Mean Methods

SES sits between the mean method and the naive method:

- **Mean method** ($\alpha \to 0$): equal weight to all observations.
- **Naive method** ($\alpha = 1$): all weight on the last observation.
- **SES** ($0 < \alpha < 1$): exponentially decaying weights — a compromise.

Let us compare their accuracy on the test set.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Mean method forecast
fc_mean = np.full(HORIZON, train["Births"].mean())

# Naive method forecast
fc_naive = np.full(HORIZON, train["Births"].iloc[-1])

# SES forecast (optimized)
fc_ses = fit.forecast(HORIZON).values

actual = test["Births"].values

comparison = pd.DataFrame({
    "Method": ["Mean", "Naive", "SES (optimized)"],
    "MAE": [
        mean_absolute_error(actual, fc_mean),
        mean_absolute_error(actual, fc_naive),
        mean_absolute_error(actual, fc_ses),
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(actual, fc_mean)),
        np.sqrt(mean_squared_error(actual, fc_naive)),
        np.sqrt(mean_squared_error(actual, fc_ses)),
    ],
})
comparison["MAE"] = comparison["MAE"].round(2)
comparison["RMSE"] = comparison["RMSE"].round(2)
comparison = comparison.sort_values("MAE").reset_index(drop=True)
print(comparison.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train["Births"].iloc[-60:], color="black", alpha=0.4, label="Train")
ax.plot(test["Births"], color="tab:blue", linewidth=2, label="Actual (test)")
ax.plot(test.index, fc_mean, linestyle="--", color="tab:red", label="Mean")
ax.plot(test.index, fc_naive, linestyle="--", color="tab:green", label="Naive")
ax.plot(test.index, fc_ses, linestyle="--", color="tab:purple", linewidth=2, label="SES")
ax.set_title("SES vs Naive vs Mean — Daily Births")
ax.set_ylabel("Number of Births")
ax.legend()
plt.tight_layout()
plt.show()

print("All three methods produce flat forecasts.")
print("SES finds a level between the mean and the naive forecast.")

---
## 11. Why SES Produces Flat Forecasts

Because SES has only a **level** component ($\ell_t$) and no trend or
seasonal component, the forecast equation is:

$$
\hat{y}_{t+h|t} = \ell_t \quad \text{for all } h = 1, 2, 3, \ldots
$$

This means:
- The 1-step-ahead forecast = the 10-step-ahead forecast = the 100-step-ahead forecast.
- SES provides **no information about the future trajectory** — it only
  estimates the current level.
- For data with trend or seasonality, we need Holt's method or Holt-Winters
  (next notebooks).

In [ ]:
# Show that extending the horizon doesn't change the forecast
fc_short = fit.forecast(10)
fc_long = fit.forecast(100)

print(f"10-step forecast:  {fc_short.values}")
print(f"\n100-step forecast (first 10): {fc_long.values[:10]}")
print(f"\nAll values identical? {np.allclose(fc_short.values, fc_long.values[:10])}")

---
## 12. When SES Fails — Trending Data

SES is **not** appropriate when the data has a trend.  Let us demonstrate
this on the airline passengers data to motivate Holt's method (next
notebook).

In [ ]:
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

air_train = airline.iloc[:120]
air_test = airline.iloc[120:]

ses_air = SimpleExpSmoothing(
    air_train["Passengers"],
    initialization_method="estimated",
).fit(optimized=True)

fc_air = ses_air.forecast(len(air_test))

print(f"Alpha on airline data: {ses_air.params['smoothing_level']:.4f}")
print(f"SES forecast (flat): {fc_air.iloc[0]:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(air_train["Passengers"], label="Train", color="black", alpha=0.5)
ax.plot(air_test["Passengers"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(fc_air, label="SES forecast", color="tab:red", linestyle="--", linewidth=2)
ax.set_title("SES on Airline Passengers — Fails on Trending + Seasonal Data")
ax.set_ylabel("Thousands of Passengers")
ax.legend()
plt.tight_layout()
plt.show()

print("SES produces a flat line that completely misses the upward trend")
print("and the seasonal pattern.  We need Holt's method (next notebook)")
print("to handle the trend, and Holt-Winters for trend + seasonality.")

---
## 13. The Level Equation as an Error-Correction Form

The level equation can be rewritten in a revealing **error-correction form**:

$$
\ell_t = \ell_{t-1} + \alpha(y_t - \ell_{t-1})
= \ell_{t-1} + \alpha \cdot e_t
$$

where $e_t = y_t - \ell_{t-1}$ is the one-step-ahead forecast error.

This shows that the level is updated by adding a fraction $\alpha$ of the
latest forecast error.  If the forecast was too low ($e_t > 0$), the level
is adjusted upward; if too high ($e_t < 0$), downward.

In [ ]:
# Demonstrate the error correction form
alpha_demo = 0.3
levels_demo = ses_from_scratch(y_train[:20], alpha_demo)

print(f"{'t':>3}  {'y_t':>8}  {'l_{t-1}':>8}  {'e_t':>8}  {'alpha*e_t':>10}  {'l_t':>8}")
print("-" * 55)
for t in range(1, 15):
    y_t = y_train[t]
    l_prev = levels_demo[t - 1]
    e_t = y_t - l_prev
    correction = alpha_demo * e_t
    l_t = levels_demo[t]
    print(f"{t:3d}  {y_t:8.1f}  {l_prev:8.2f}  {e_t:8.2f}  {correction:10.2f}  {l_t:8.2f}")

print("\nThe level adjusts by alpha * forecast_error at each step.")

---
## 14. Choosing the Initial Level

The recursion $\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}$ requires a
starting value $\ell_0$.  Common choices:

1. **$\ell_0 = y_1$** — use the first observation (simple, traditional).
2. **Estimated** — treat $\ell_0$ as a parameter and optimize it alongside
   $\alpha$ (the approach used by `initialization_method='estimated'`).
3. **$\ell_0 = \bar{y}_{\text{first few}}$** — average of the first few
   observations (a compromise).

For short series the choice matters; for long series it fades quickly
because the initial level's influence decays as $(1-\alpha)^t$.

In [ ]:
# Show how quickly the initial level's influence decays
fig, ax = plt.subplots(figsize=(10, 5))

for alpha in [0.1, 0.3, 0.5]:
    influence = (1 - alpha) ** np.arange(50)
    ax.plot(influence, label=f"alpha={alpha}")

ax.set_title("Influence of Initial Level Over Time: $(1-\\alpha)^t$")
ax.set_xlabel("Time steps since initialization")
ax.set_ylabel("Weight on initial level")
ax.axhline(0.01, color="grey", linestyle="--", alpha=0.5, label="1% threshold")
ax.legend()
plt.tight_layout()
plt.show()

print("For alpha=0.1, the initial level still has >1% influence after 40 steps.")
print("For alpha=0.5, the influence drops below 1% after about 7 steps.")

In [ ]:
# Compare different initial levels — they converge
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, y_train, color="black", alpha=0.3, label="Actual")

for l0 in [20, 35, 50, 70]:
    levels = ses_from_scratch(y_train, alpha=0.3, initial_level=l0)
    ax.plot(train.index, levels, label=f"l0 = {l0}", linewidth=1.2)

ax.set_title("SES with Different Initial Levels (alpha=0.3)")
ax.set_ylabel("Level")
ax.legend()
plt.tight_layout()
plt.show()

print("Regardless of the initial level, the smoothed values converge quickly.")
print("This is why the choice of l_0 rarely matters for long series.")

---
## Summary

| Concept | Detail |
|---------|--------|
| **What** | Weighted average forecast with exponentially decaying weights |
| **When to use** | Data with no trend and no seasonality |
| **Parameters** | $\alpha$ (smoothing), $\ell_0$ (initial level) |
| **Forecast shape** | Flat — all horizons get the same value $\ell_T$ |
| **$\alpha$ near 0** | Slow adaptation, smooth level, long memory |
| **$\alpha$ near 1** | Fast adaptation, reactive level, short memory (approaches naive) |
| **Optimization** | Minimize SSE (or maximize likelihood) to find best $\alpha$ and $\ell_0$ |
| **Limitation** | Cannot handle trends or seasonality |

**Next notebook**: Holt's Method — extending SES with a trend component.

In [ ]:
print("Next notebook: Holt's Method — adding a trend component to SES")
print("to handle data with upward or downward trends.")